ERA5 reanalysis atmospheres
===========================

MCEq can run on a **measured or reanalysed atmosphere** instead of a
parametrization, through
`MCEq.geometry.density_profiles.TabulatedAtmosphere` and its location-centred
sibling `TabulatedLocationCentered`. They read a CSV table; converting a
dataset into that table is left to the user, because every archive has its own
API and file format and MCEq does not want to depend on any of them.

This notebook is the worked example for **ERA5**, the ECMWF reanalysis. It

1. converts an ERA5 netCDF download into MCEq tables — one per day, each a
   global longitude/latitude grid of vertical columns,
2. runs MCEq on them and compares against NRLMSISE-00,
3. maps and animates the result globally, with every plotted number coming
   out of the MCEq atmosphere interface rather than straight from the file,
4. shows why the grid matters for neutrinos: at large zenith angles, and for
   upgoing events in particular, the shower develops in a completely different
   column than the one above the detector.

> **Not executed when the documentation is built.** It needs a CDS account and
> a multi-hundred-MB download. Run it locally.

What you need
-------------

```bash
pip install cdsapi xarray netcdf4 cartopy
```

(cartopy downloads Natural Earth coastline data the first time it draws
a map, so the first plot needs a network connection.)

and, for the download, a CDS account with an API key in `~/.cdsapirc`
(register at <https://cds.climate.copernicus.eu>, then follow
<https://cds.climate.copernicus.eu/how-to-api> and accept the dataset licence
once from its download page).

The table format
----------------

A table is a CSV file. In its simplest form it is **one vertical column**:

```
# MCEq tabulated atmosphere v1
h_cm,T_K,p_hPa
283400.0,247.1,681.2
510000.0,231.4,500.0
900000.0,,300.0
```

* `h_cm` is required: height above sea level in cm.
* Density comes either from a `rho_gcm3` column directly, or from `T_K` and
  `p_hPa` via the dry-air ideal gas law.
* `T_K` also feeds `get_temperature()`, `p_hPa` feeds `get_pressure()`.
* Comment lines start with `#`, column order does not matter, unknown columns
  are ignored, rows may be in any order, and missing values are an empty field
  or `nan`.

Adding **`lat_deg` and `lon_deg`** turns it into a *grid* of columns — one row
per (grid node, level), long format:

```
lat_deg,lon_deg,h_cm,T_K,p_hPa
-90.0,0.0,110.9,242.8,1000
-90.0,0.0,1352.0,241.6,975
...
```

The nodes must form a regular longitude/latitude grid, and every column must
carry the same number of levels. That is exactly how ERA5 pressure-level data
is shaped, so the conversion is a reshape.

In [ ]:
import os

import numpy as np
import xarray as xr

# --- input ----------------------------------------------------------------
# An ERA5 pressure-level download. This notebook was written against
# "ERA5 daily statistics on pressure levels" (daily mean), variables
# temperature (+ geopotential, see below), all 37 levels, global.
NC_FILE = "era5_daily_pressure_levels.nc"

# Full 0.25 deg is 721 x 1440 = 1.04 M columns; a CSV of that is 38 M rows per
# day. Coarsen for the example. STRIDE 16 -> 4 deg, 46 x 90 = 4140 columns.
STRIDE = 16

OUTPUT_DIR = "era5_tables"

# --- the detector ---------------------------------------------------------
# KM3NeT/ARCA: off Capo Passero, and far enough from the pole that the azimuth
# angle genuinely changes which column the shower develops in.
SITE_NAME = "KM3NeT-ARCA"
SITE_LON, SITE_LAT = 16.1, 36.267
SITE_DEPTH_M = 3500.0
SITE_ELEVATION_M = 0.0

In [ ]:
ds = xr.open_dataset(NC_FILE, chunks={"valid_time": 1})
print(ds)

HAS_GEOPOTENTIAL = "z" in ds.data_vars
print(f"\ngeopotential in file: {HAS_GEOPOTENTIAL}")

### Heights

MCEq integrates along a slant path through *geometric height*, so every level
needs one. There are two ways to get it, and which one you have depends on what
you asked the CDS for.

**With `geopotential` in the file** it is exact and immediate:

$$ h = \frac{z}{g_0}, \qquad g_0 = 9.80665\ \mathrm{m/s^2} $$

**Without it** the heights have to be rebuilt from the temperatures with the
hypsometric equation,

$$ \Delta z = \frac{R_d}{g_0}\,\bar{T}\,\ln\frac{p_\mathrm{low}}{p_\mathrm{up}}, $$

integrated upward from an anchor. The *thicknesses* it gives are good — they use
the real temperatures — but the anchor is a guess: this code puts the 1000 hPa
surface at its US-Standard height of 111 m everywhere, whereas in reality it
moves by ±150 m with the weather and, over terrain higher than that, is below
ground and fictitious. That error is a rigid shift of the whole column above it.

It barely touches the density profile, which comes from $p$ and $T$ alone and
needs no height at all, but it does move the profile up or down against the
geometry. **Add `geopotential` to the CDS request if you can.** The converter
below uses it when present and falls back to the integration when not.

In [ ]:
G0 = 9.80665  # standard gravity, m/s^2
R_D = 287.06  # gas constant of dry air, J/(kg K)
H_1000_HPA_M = 110.9  # US Standard height of the 1000 hPa surface


def column_heights(pressure_hpa, temperature_k, geopotential=None):
    """Geometric height of every pressure level, in metres.

    Args:
      pressure_hpa: (n_lev,) levels, ascending in height (descending pressure)
      temperature_k: (..., n_lev) temperatures
      geopotential: (..., n_lev) geopotential in m^2/s^2, or None

    Returns:
      (..., n_lev) heights above sea level in metres
    """
    if geopotential is not None:
        return geopotential / G0

    # Hypsometric integration upward from the bottom level.
    thickness = (
        R_D
        / G0
        * 0.5
        * (temperature_k[..., 1:] + temperature_k[..., :-1])
        * np.log(pressure_hpa[:-1] / pressure_hpa[1:])
    )
    base = np.full(temperature_k.shape[:-1] + (1,), H_1000_HPA_M)
    return np.concatenate([base, H_1000_HPA_M + np.cumsum(thickness, axis=-1)], axis=-1)


def write_gridded_table(filename, lat, lon, h_cm, t_k, p_hpa, comment):
    """Writes a v1 MCEq tabulated-atmosphere CSV holding a grid of columns.

    Args:
      lat: (n_lat,) latitudes, lon: (n_lon,) longitudes
      h_cm, t_k: (n_lat, n_lon, n_lev); p_hpa: (n_lev,)
    """
    n_lat, n_lon, n_lev = h_cm.shape
    lat_col = np.repeat(lat, n_lon * n_lev)
    lon_col = np.tile(np.repeat(lon, n_lev), n_lat)
    p_col = np.tile(p_hpa, n_lat * n_lon)
    rows = np.column_stack(
        [lat_col, lon_col, h_cm.reshape(-1), t_k.reshape(-1), p_col]
    )
    with open(filename, "w") as out:
        out.write("# MCEq tabulated atmosphere v1\n")
        out.write(f"# {comment}\n")
        out.write("lat_deg,lon_deg,h_cm,T_K,p_hPa\n")
        np.savetxt(out, rows, delimiter=",", fmt="%.4f,%.4f,%.6e,%.3f,%.6g")
    return filename

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

sub = ds.isel(latitude=slice(None, None, STRIDE), longitude=slice(None, None, STRIDE))
lat = sub["latitude"].values
lon = sub["longitude"].values

# Order the levels by ascending height, i.e. descending pressure.
p_hpa = np.sort(sub["pressure_level"].values)[::-1]

table_files = []
for step in range(sub.sizes["valid_time"]):
    day = sub.isel(valid_time=step)
    date = str(day["valid_time"].values)[:10]

    t_k = (
        day["t"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
    )
    z = (
        day["z"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
        if HAS_GEOPOTENTIAL
        else None
    )
    h_cm = column_heights(p_hpa, t_k, z) * 1e2

    path = write_gridded_table(
        os.path.join(OUTPUT_DIR, f"era5_{date}.csv"),
        lat, lon, h_cm, t_k, p_hpa,
        f"ERA5 daily mean, {date}, {lat.size}x{lon.size} grid, "
        f"heights from {'geopotential' if HAS_GEOPOTENTIAL else 'hypsometric integration'}",
    )
    table_files.append(path)
    print(f"{path}  {os.path.getsize(path) / 1e6:5.1f} MB  "
          f"top level {h_cm[..., -1].mean() / 1e5:.1f} km (mean)")

### One site

`("Tabulated", (path, coord))` picks a single column out of the grid;
`("Tabulated_LC", (path, coord, depth_m))` binds the grid to a detector and
samples it at the shower impact point. Start with the plain column and compare
it to NRLMSISE-00 at the same place and season.

In [ ]:
import matplotlib.pyplot as plt

import MCEq.geometry.density_profiles as dp

table = dp.load_atmosphere_table(table_files[0])
print(table)

era5_atm = dp.TabulatedAtmosphere(
    table, coord=(SITE_LON, SITE_LAT), location=SITE_NAME, season="July"
)
msis_atm = dp.MSIS00LocationCentered(
    detector_coord=(SITE_LON, SITE_LAT), depth_m=SITE_DEPTH_M, season="July"
)
for atm in (era5_atm, msis_atm):
    atm.set_theta(0.0)
print(f"ERA5 vertical column {era5_atm.max_X:8.2f} g/cm^2")
print(f"MSIS vertical column {msis_atm.max_X:8.2f} g/cm^2")

#### Profiles against pressure

Without geopotential the heights carry the anchor uncertainty described above,
so the honest vertical coordinate for a comparison is **slant depth**, which
MCEq owns and which for a vertical column is just $X \simeq p/g$. Everything
below is read back out of the MCEq interface: `X2rho`, `X2h`, `get_temperature`.

Once the file has geopotential, swap the y-axis for `X2h(X)/1e5` and the same
code plots against height.

In [ ]:
X = np.geomspace(1.0, min(era5_atm.max_X, msis_atm.max_X) * 0.999, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), dpi=120, sharey=True)

for atm, label, style in ((era5_atm, "ERA5", "-"), (msis_atm, "MSIS00", "--")):
    rho = atm.X2rho(X)
    temp = np.array([float(atm.get_temperature(h)) for h in atm.X2h(X)])
    axes[0].plot(rho, X, style, label=label)
    axes[1].plot(temp, X, style, label=label)

axes[2].plot(
    era5_atm.X2rho(X) / msis_atm.X2rho(X), X, "-", label=r"$\rho$"
)
axes[2].plot(
    np.array([float(era5_atm.get_temperature(h)) for h in era5_atm.X2h(X)])
    / np.array([float(msis_atm.get_temperature(h)) for h in msis_atm.X2h(X)]),
    X, "--", label="T",
)
axes[2].axvline(1.0, color="0.6", lw=0.8)

axes[0].set_xscale("log")
axes[0].set_xlabel(r"$\rho$ [g/cm$^3$]")
axes[1].set_xlabel("T [K]")
axes[2].set_xlabel("ERA5 / MSIS00")
axes[0].set_ylabel(r"vertical depth $X$ [g/cm$^2$]  ($\simeq p/g$)")
for ax in axes:
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"{SITE_NAME}, {str(ds['valid_time'].values[0])[:10]}")
fig.tight_layout()

### Global maps, through the MCEq interface

Both quantities below are read out of an MCEq atmosphere object built for each
grid node — not taken from the netCDF. That is the point: it exercises the same
code path a flux calculation uses, so what the map shows is what MCEq would
integrate.

* **temperature at a fixed slant depth**,
  `atm.get_temperature(atm.X2h(X))`. At $X = 100$ g/cm² that is the lower
  stratosphere, where the mesons that make atmospheric leptons are produced —
  the quantity whose seasonal swing drives the muon and neutrino rate
  variations IceCube and KM3NeT measure. At $X = 700$ g/cm² it is the lower
  troposphere, i.e. weather.
* **vertical column depth** `atm.max_X`, the whole atmosphere in g/cm².

A caveat on the second one while the file has no geopotential: the hypsometric
reconstruction pins every column to the same 111 m anchor at 1000 hPa, so
`max_X` comes out at 1019.x g/cm² everywhere and its map shows only the
isothermal tail wobbling. Real surface pressure varies by tens of g/cm² over
weather systems and by hundreds over terrain. **That map becomes meaningful the
moment `geopotential` is in the download** — which is the single best reason to
add it.

In [ ]:
DEPTHS = (700.0, 100.0)  # lower troposphere, lower stratosphere [g/cm^2]


def mceq_maps(table, depths=DEPTHS):
    """Column depth and temperatures at fixed slant depths, via MCEq.

    One TabulatedAtmosphere per grid node; every number comes from max_X and
    get_temperature(X2h(...)), i.e. from the same interface the solver uses.

    Returns:
      (max_X, temps) with temps shaped (len(depths), n_lat, n_lon)
    """
    n_lat, n_lon = table.lat_deg.size, table.lon_deg.size
    max_X = np.empty((n_lat, n_lon))
    temps = np.empty((len(depths), n_lat, n_lon))
    for j, node_lat in enumerate(table.lat_deg):
        for i, node_lon in enumerate(table.lon_deg):
            atm = dp.TabulatedAtmosphere(table, coord=(node_lon, node_lat))
            atm.set_theta(0.0)
            max_X[j, i] = atm.max_X
            for k, depth in enumerate(depths):
                temps[k, j, i] = atm.get_temperature(atm.X2h(depth))
    return max_X, temps


import time

start = time.time()
max_X, temps = mceq_maps(table)
print("%d columns through MCEq in %.1f s" % (max_X.size, time.time() - start))
for depth, field in zip(DEPTHS, temps):
    print("T at %4.0f g/cm^2  %.1f .. %.1f K" % (depth, field.min(), field.max()))
print("column depth     %.1f .. %.1f g/cm^2  <- flat until the file has "
      "geopotential" % (max_X.min(), max_X.max()))

In [ ]:
import warnings

import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

# Cartopy clips coastline polygons at the edge of the Robinson projection and
# shapely grumbles about the empty geometries that produces. Harmless, loud.
warnings.filterwarnings(
    "ignore", message="invalid value encountered in create_collection"
)


def global_map(ax, field, lat, lon, cmap, label, **kwargs):
    """Draws a lon/lat field on a Robinson projection."""
    cyclic, lon_c = add_cyclic_point(field, coord=lon)
    mesh = ax.pcolormesh(
        lon_c, lat, cyclic, transform=ccrs.PlateCarree(),
        cmap=cmap, shading="auto", **kwargs
    )
    ax.coastlines(lw=0.4, color="0.3")
    ax.set_global()
    plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.04, label=label)
    return mesh


fig, axes = plt.subplots(
    1, 2, figsize=(13, 4.5), dpi=120,
    subplot_kw={"projection": ccrs.Robinson()},
)
for ax, depth, field in zip(axes, DEPTHS, temps):
    global_map(ax, field, table.lat_deg, table.lon_deg, "RdBu_r",
               f"T at {depth:.0f} g/cm$^2$ [K]")
axes[0].plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
axes[1].plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
fig.suptitle(f"ERA5 through MCEq — {str(ds['valid_time'].values[0])[:10]}")
fig.tight_layout()

#### Day to day

The same map for every day in the download. Even over one week the
stratospheric temperature field moves noticeably, and that is what a seasonal
variation analysis is chasing.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

frames = []
for path in table_files:
    day_table = dp.load_atmosphere_table(path)
    frames.append((path, mceq_maps(day_table)[1][1]))
    print("done", path)

stack = np.array([f[1] for f in frames])
vmin, vmax = np.percentile(stack, [1, 99])

fig = plt.figure(figsize=(8, 4.5), dpi=110)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
mesh = global_map(ax, frames[0][1], table.lat_deg, table.lon_deg, "RdBu_r",
                  "T at 100 g/cm$^2$ [K]", vmin=vmin, vmax=vmax)
title = ax.set_title("")


def draw(step):
    path, temp = frames[step]
    cyclic, _ = add_cyclic_point(temp, coord=table.lon_deg)
    mesh.set_array(cyclic.ravel())
    title.set_text(os.path.basename(path).replace("era5_", "").replace(".csv", ""))
    return mesh, title


ani = animation.FuncAnimation(fig, draw, frames=len(frames), interval=700, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

### Why the grid matters for neutrinos

`TabulatedLocationCentered` follows the shower axis from the detector toward
the source and takes the table column where it crosses the surface. For a
downgoing shower that is near the detector. For an **upgoing** one — the signal
region of a neutrino telescope — the axis passes through the Earth and the
shower developed on the *far side*, in an atmosphere that has nothing to do
with the one overhead. A single-column table cannot express that; a global grid
can, which is why `max_theta=180` needs one.

The map below traces the impact point over the whole sky, coloured by the slant
depth MCEq computes there.

In [ ]:
lc_atm = dp.TabulatedLocationCentered(
    table,
    detector_coord=(SITE_LON, SITE_LAT),
    depth_m=SITE_DEPTH_M,
    max_theta=180.0,
    location=SITE_NAME,
    season="July",
)

zeniths = np.arange(0.0, 180.1, 7.5)
azimuths = np.arange(0.0, 360.0, 45.0)
track = []
for azimuth in azimuths:
    for zenith in zeniths:
        lc_atm.set_theta(float(zenith), azimuth_deg=float(azimuth))
        track.append(
            (lc_atm.current_impact_longitude, lc_atm.current_impact_latitude,
             lc_atm.max_X, zenith, azimuth)
        )
track = np.array(track)
print(f"{len(track)} directions, slant depth "
      f"{track[:, 2].min():.0f} .. {track[:, 2].max():.0f} g/cm^2")

In [ ]:
fig = plt.figure(figsize=(13, 5.2), dpi=120)
downgoing = track[:, 3] <= 90.0

# --- the whole sky ---------------------------------------------------------
ax = fig.add_subplot(1, 2, 1, projection=ccrs.Robinson())
global_map(ax, temps[1], table.lat_deg, table.lon_deg, "RdBu_r",
           f"T at {DEPTHS[1]:.0f} g/cm$^2$ [K]", alpha=0.45)
for mask, marker, name in (
    (downgoing, "o", "downgoing"),
    (~downgoing, "^", "upgoing (far side)"),
):
    scat = ax.scatter(
        track[mask, 0], track[mask, 1], c=np.log10(track[mask, 2]),
        s=22, marker=marker, cmap="viridis", vmin=np.log10(track[:, 2]).min(),
        vmax=np.log10(track[:, 2]).max(), transform=ccrs.PlateCarree(),
        zorder=3, edgecolors="k", linewidths=0.25, label=name,
    )
ax.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
plt.colorbar(scat, ax=ax, orientation="vertical", pad=0.02,
             label=r"$\log_{10}$ slant depth [g/cm$^2$]")
ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"All sky from {SITE_NAME} (star):\nupgoing showers develop on the far side", fontsize=10)

# --- and the downgoing cluster, which the star hides above ------------------
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
near = ax2.scatter(
    track[downgoing, 0], track[downgoing, 1], c=track[downgoing, 3],
    s=26, cmap="viridis", transform=ccrs.PlateCarree(), zorder=3,
    edgecolors="k", linewidths=0.25,
)
ax2.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
ax2.set_extent([SITE_LON - 3.5, SITE_LON + 3.5, SITE_LAT - 3.0, SITE_LAT + 3.0],
               crs=ccrs.PlateCarree())
ax2.coastlines(lw=0.5, color="0.3")
grid = ax2.gridlines(draw_labels=True, lw=0.3, color="0.7")
grid.top_labels = grid.right_labels = False
plt.colorbar(near, ax=ax2, orientation="vertical", pad=0.02,
             label="zenith angle [deg]")
ax2.set_title("Downgoing: the column drifts\nwith zenith and azimuth", fontsize=10)

# tight_layout does not cope with cartopy gridline labels
fig.subplots_adjust(left=0.02, right=0.97, top=0.86, wspace=0.28)

### Where to go from here

* **Add geopotential** to the CDS request and the heights become exact; the
  profile plots can then use `X2h(X)` on the y-axis directly.
* **Resolution.** `STRIDE = 16` is 4 deg, about 440 km. The impact point moves
  ~100 km by 80 deg zenith, so drop the stride for a real analysis and crop to
  the region you need — CSV is a text format and the full 0.25 deg grid is 38 M
  rows per day.
* **Seasonal studies.** One table per day; build one atmosphere per table and
  pass them to `MCEqRun.solve_batch(..., conditions=...)` as per-member
  `density_model`s to get a time series in one call.
* **Other datasets.** Only the converter above is ERA5-specific. Write
  `h_cm` plus `T_K`/`p_hPa` (or `rho_gcm3`), optionally `lat_deg`/`lon_deg`,
  and any archive works.